In [33]:
import pandas as pd
import pathlib as path
from sklearn.model_selection import train_test_split

import pandas as pd
from pathlib import Path

In [34]:
df = pd.read_csv('../DataSet/Telco-Customer-Churn.csv')
if not df.empty:
    print("DataFrame is not empty")
else:
    print("DataFrame is empty")

print("Dataset loaded successfully.")

DataFrame is not empty
Dataset loaded successfully.


In [35]:
total_charges_numeric = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

missing_total_charges = (
    total_charges_numeric.isna()
)

print(
    "Blank or invalid TotalCharges values:",
    missing_total_charges.sum()
)

Blank or invalid TotalCharges values: 0


In [36]:
df_clean = df.copy()
cleaning_summary = pd.DataFrame({
    "Check": [
        "Rows",
        "Columns",
        "Missing values",
        "Duplicate rows",
        "Unique customer IDs"
    ],
    "Result": [
        df_clean.shape[0],
        df_clean.shape[1],
        df_clean.isna().sum().sum(),
        df_clean.duplicated().sum(),
        df_clean["customerID"].nunique()
    ]
})

In [37]:
cleaning_summary

,Check,Result
0,Rows,7032
1,Columns,21
2,Missing values,0
3,Duplicate rows,0
4,Unique customer IDs,7032


In [38]:
assert df_clean.shape == (7032, 21)
assert df_clean.isna().sum().sum() == 0
assert df_clean.duplicated().sum() == 0
assert df_clean["customerID"].nunique() == 7032

print("All cleaning checks passed.")

All cleaning checks passed.


In [39]:
df_clean = df.dropna()
total_charges_numeric = pd.to_numeric(
    df_clean["TotalCharges"],
    errors="coerce"
)

missing_total_charges = total_charges_numeric.isna()
assert df_clean.loc[
    missing_total_charges,
    "tenure"
].eq(0).all()
df_clean["TotalCharges"] = total_charges_numeric.fillna(0)

df_clean = df_clean.drop_duplicates().copy()
print("Cleaned shape:", df_clean.shape)
print("Missing values:", df_clean.isna().sum().sum())

Cleaned shape: (7032, 21)
Missing values: 0


In [40]:
RANDOM_STATE = 50

# Remove the identifier and separate the target
X = df_clean.drop(
    columns=["customerID", "Churn"]
).copy()

y = df_clean["Churn"].map({
    "No": 0,
    "Yes": 1
})

assert y.notna().all()
assert "customerID" not in X.columns
assert "Churn" not in X.columns

In [41]:
# 70% training and 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# Divide temporary data into 15% validation and 15% test
X_validation, X_test, y_validation, y_test = (
    train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=RANDOM_STATE
    )
)

split_summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Test"
    ],
    "Rows": [
        len(y_train),
        len(y_validation),
        len(y_test)
    ],
    "Churned Customers": [
        y_train.sum(),
        y_validation.sum(),
        y_test.sum()
    ],
    "Churn Rate (%)": [
        y_train.mean() * 100,
        y_validation.mean() * 100,
        y_test.mean() * 100
    ]
})

In [42]:
split_summary.round(2)

,Dataset,Rows,Churned Customers,Churn Rate (%)
0,Training,4922,1308,26.57
1,Validation,1055,281,26.64
2,Test,1055,280,26.54


In [43]:
assert X_train.index.intersection(
    X_validation.index
).empty

assert X_train.index.intersection(
    X_test.index
).empty

assert X_validation.index.intersection(
    X_test.index
).empty

assert (
    len(X_train)
    + len(X_validation)
    + len(X_test)
) == len(X)

print("Split verification passed.")

Split verification passed.


In [44]:
dataset_folder = Path("../DataSet")

if not dataset_folder.exists():
    dataset_folder = Path("DataSet")

processed_folder = dataset_folder / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

train_data = X_train.copy()
train_data["Churn"] = y_train
train_data = train_data.reset_index(drop=True)

validation_data = X_validation.copy()
validation_data["Churn"] = y_validation
validation_data = validation_data.reset_index(drop=True)

test_data = X_test.copy()
test_data["Churn"] = y_test
test_data = test_data.reset_index(drop=True)

train_data.to_csv(
    processed_folder / "telco_churn_train.csv",
    index=False
)

validation_data.to_csv(
    processed_folder / "telco_churn_validation.csv",
    index=False
)

test_data.to_csv(
    processed_folder / "telco_churn_test.csv",
    index=False
)

print("Dataset splits saved to:")
print(processed_folder.resolve())





Dataset splits saved to:
/Users/yadanarlin/Desktop/Telecommunication-Customer-Churn-Prediction/DataSet/processed
